# Few-Shot 피드백 생성기 (이형 스타일)

vertex_ai.ipynb에서 했던 Few-Shot Prompting을 Gemini API로 재구현.

**원리:** 실제 면접 인터뷰+피드백 쌍을 예시로 보여주고,  
새 답변에도 같은 스타일로 피드백하도록 유도.

**파인튜닝과 차이:** 모델 가중치는 안 바꾸고, 대화 컨텍스트에 예시를 넣는 방식.

## 실행 순서
1. STEP 1~3 순서대로 실행 (환경 설정)
2. STEP 4에서 Few-Shot 예시 개수 설정
3. STEP 5에서 질문/답변 입력 후 피드백 받기

In [ ]:
%pip install google-genai python-dotenv --quiet

---
## STEP 1 — 환경 설정

In [ ]:
import os, json, time, re
from pathlib import Path
from dotenv import load_dotenv

BASE      = Path(r'C:\Users\82105\OneDrive\바탕 화면\interview-coach')
DATA_DIR  = BASE / 'data' / 'feedback_data'

load_dotenv(BASE.parent / '프로젝트3(면접)' / '.env')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
if not GOOGLE_API_KEY:
    raise ValueError('.env 파일에 GOOGLE_API_KEY가 없습니다.')

# Few-Shot에는 context window가 넉넉한 모델 필요
# gemini-flash-lite는 영상/긴 컨텍스트 미지원 → 1.5-flash 사용
MODEL = 'gemini-1.5-flash'

print(f'설정 완료 | 모델: {MODEL}')
print(f'데이터 경로: {DATA_DIR}')

---
## STEP 2 — 인터뷰+피드백 쌍 데이터 로드

In [ ]:
pairs = []  # {'interview': str, 'feedback': str}

# ── 1~5: Interview_text / Feedback_text 폴더로 분리 ──────────────────────
interview_dir = DATA_DIR / '1~5' / 'Interview_text'
feedback_dir  = DATA_DIR / '1~5' / 'Feedback_text'

for iv_file in sorted(interview_dir.glob('*.txt')):
    # video_1.txt → feedback도 video_1.txt
    fb_file = feedback_dir / iv_file.name
    if fb_file.exists():
        iv_text = iv_file.read_text(encoding='utf-8').strip()
        fb_text = fb_file.read_text(encoding='utf-8').strip()
        if iv_text and fb_text:
            pairs.append({'interview': iv_text, 'feedback': fb_text,
                          'source': iv_file.name})

# ── 11~16: interview_text / feedback_text 폴더로 분리 ────────────────────
interview_dir2 = DATA_DIR / '11~16' / 'interview_text'
feedback_dir2  = DATA_DIR / '11~16' / 'feedback_text'

for iv_file in sorted(interview_dir2.glob('*.txt')):
    fb_file = feedback_dir2 / iv_file.name
    if fb_file.exists():
        iv_text = iv_file.read_text(encoding='utf-8').strip()
        fb_text = fb_file.read_text(encoding='utf-8').strip()
        if iv_text and fb_text:
            pairs.append({'interview': iv_text, 'feedback': fb_text,
                          'source': iv_file.name})

# ── 6~10: A./F. 마커로 혼합된 파일 파싱 ─────────────────────────────────
text_dir = DATA_DIR / '6~10' / 'text'
interview_files = sorted(text_dir.glob('*인터뷰*.txt'))

for iv_file in interview_files:
    # 인터뷰 파일명에서 번호 추출: '6_인터뷰 text.txt' → '6'
    num = re.match(r'(\d+)', iv_file.name)
    if not num:
        continue
    n = num.group(1)
    iv_text = iv_file.read_text(encoding='utf-8').strip()

    # 같은 번호의 피드백 파일들 합치기 (6_1.txt, 6_2.txt ...)
    fb_files = sorted(text_dir.glob(f'{n}_[0-9]*.txt'))
    fb_text = '\n\n'.join(f.read_text(encoding='utf-8').strip()
                           for f in fb_files if f.exists())
    if iv_text and fb_text:
        pairs.append({'interview': iv_text, 'feedback': fb_text,
                      'source': iv_file.name})

print(f'로드 완료: {len(pairs)}개 인터뷰-피드백 쌍')
print(f'\n첫 번째 샘플:')
print(f'  [인터뷰] {pairs[0]["interview"][:80]}...')
print(f'  [피드백] {pairs[0]["feedback"][:80]}...')

---
## STEP 3 — Gemini 클라이언트 + Few-Shot contents 구성

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

# ── Few-Shot 예시 개수 설정 ───────────────────────────────────────────────
# 많을수록 스타일 재현 정확, 토큰 소모 큼
# gemini-1.5-flash: 1M 토큰 컨텍스트 → 넉넉함
N_SHOTS = 10   # ← 여기서 조정 (최대 len(pairs))

SYSTEM_PROMPT = """당신은 대기업 인사담당자 출신의 직설적인 면접 코치 '이형'이다.
10년간 수천 명을 면접했고, 지원자 답변을 들으면 합격/불합격이 바로 보인다.
친근한 말투를 쓰되 평가는 냉정하게 한다.
아래 형식으로만 답한다:

[이형의 팩폭 한줄평]
한 문장으로 이 답변의 핵심 문제 또는 강점을 직격한다.

[이형의 시선]
면접관 관점에서 이 답변이 어떻게 들리는지 구체적으로 분석한다. (3~5문장)

[이형의 합격 처방전]
1. 즉시 실천 가능한 구체적 개선 방법
2. 답변 구조/내용 개선 방법
3. 면접관에게 어필할 포인트"""


def build_fewshot_contents(new_answer: str, n_shots: int = N_SHOTS):
    """Few-Shot 예시 + 새 질문을 contents 리스트로 구성"""
    import random
    shots = random.sample(pairs, min(n_shots, len(pairs)))

    contents = []
    for shot in shots:
        # 예시: 인터뷰 → 피드백
        contents.append(types.Content(
            role='user',
            parts=[types.Part.from_text(
                text=f'[지원자 답변]\n{shot["interview"]}'
            )]
        ))
        contents.append(types.Content(
            role='model',
            parts=[types.Part.from_text(
                text=shot['feedback']
            )]
        ))

    # 실제 질문
    contents.append(types.Content(
        role='user',
        parts=[types.Part.from_text(
            text=f'[지원자 답변]\n{new_answer}'
        )]
    ))
    return contents


print(f'Few-Shot 설정: {N_SHOTS}개 예시 사용')
print(f'전체 가용 예시: {len(pairs)}개')

In [ ]:
def fewshot_feedback(answer: str, n_shots: int = N_SHOTS,
                     verbose: bool = True) -> str:
    """Few-Shot 방식으로 이형 스타일 피드백 생성"""
    contents = build_fewshot_contents(answer, n_shots)

    if verbose:
        print(f'Few-Shot 예시 {n_shots}개 로드 → Gemini 호출 중...')

    resp = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.4,
            max_output_tokens=1000
        )
    )

    if verbose:
        print('\n' + '━' * 60)
        print(resp.text)
        print('━' * 60)

    return resp.text


print('피드백 함수 준비 완료')

---
## STEP 4 — 피드백 받기

> **여기서 답변을 수정하고 셀을 실행하세요.**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ▼ 여기를 수정하세요
ANSWER = """안녕하세요. 저는 성실하고 책임감 있는 사람입니다.
학교에서 열심히 공부했고, 팀 프로젝트도 많이 해봤습니다.
이 회사에 오고 싶어서 지원했습니다. 잘 부탁드립니다."""

N_SHOTS = 10   # 예시 개수 조정 가능 (5~20 권장)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

result = fewshot_feedback(ANSWER, n_shots=N_SHOTS)

---
## STEP 5 — RAG vs Few-Shot 비교

같은 답변에 두 방식의 피드백을 나란히 비교.

In [ ]:
# RAG 방식 (03_feedback.ipynb와 동일)
import chromadb
from chromadb.utils import embedding_functions
from google.genai import types as gtypes

CHROMA_DIR = BASE.parent / '프로젝트3(면접)' / 'chroma_db'
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = chroma_client.get_collection('interview_rag', embedding_function=ef)

def rag_feedback(answer: str) -> str:
    results = collection.query(query_texts=[answer[:200]], n_results=4)
    contexts = '\n\n'.join(
        f'[참고{i+1}] {c}' for i, c in enumerate(results['documents'][0])
    )
    prompt = f"""{SYSTEM_PROMPT}

[지원자 답변]
{answer}

--- 참고 자료 (면접왕 이형 채널) ---
{contexts}

위 답변에 대해 이형 스타일로 피드백하라."""

    resp = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=prompt,
        config=gtypes.GenerateContentConfig(temperature=0.4, max_output_tokens=800)
    )
    return resp.text


print('═' * 60)
print('▶ RAG 방식 피드백 (gemini-flash-lite + ChromaDB)')
print('═' * 60)
rag_result = rag_feedback(ANSWER)
print(rag_result)

print('\n' + '═' * 60)
print('▶ Few-Shot 방식 피드백 (gemini-1.5-flash + 실제 예시)')
print('═' * 60)
fs_result = fewshot_feedback(ANSWER, n_shots=N_SHOTS)

---
## STEP 6 — N_SHOTS 수에 따른 품질 실험

In [ ]:
# Few-Shot 개수별 결과 비교 (선택)
TEST_ANSWER = ANSWER

for n in [1, 3, 5, 10]:
    if n > len(pairs):
        continue
    print(f'\n{"─"*60}')
    print(f'[{n}-Shot]')
    print('─' * 60)
    out = fewshot_feedback(TEST_ANSWER, n_shots=n, verbose=False)
    print(out[:300], '...' if len(out) > 300 else '')
    time.sleep(3)  # 15 RPM 한도 대비